In [24]:
# Importando bibliotecas
from functions import *
import pandas as pd
import locale
from pathlib import Path
import shutil
from datetime import datetime
import warnings
import logging
from openpyxl import load_workbook

timer = Temporizador()
timer.iniciar()

locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')  # Para Windows
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.expand_frame_repr', False)

# Detecta se o script está sendo executado de um .py ou de um notebook
try:
    caminho_base = Path(__file__).resolve().parent
except NameError:
    # __file__ não existe em Jupyter ou ambiente interativo
    caminho_base = Path.cwd()

pasta_input_parquet = caminho_base.parent / '01_INPUT_PIPELINE/01_BD_PARQUET'
pasta_staging_parquet = caminho_base.parent / '02_STAGING_PARQUET'
pasta_painel = caminho_base.parent / '05_PAINEL'
pasta_historico_planos = caminho_base.parent / '04_HISTORICO_PLANOS'
arquivo_input_regras_negocio = caminho_base.parent / '01_INPUT_PIPELINE/02_REGRAS_NEGOCIO/KRONA_REGRAS.xlsm'
origem = pasta_painel / 'PREV_DEMANDA_KRONA.xlsb'
copia = pasta_painel / 'PREV_DEMANDA_KRONA_TEMP.xlsb'

print("✅ Mapeamento de pastas concluído com sucesso!")

✅ Mapeamento de pastas concluído com sucesso!


In [18]:
# FIXME: apenas para gerar uma saída pra Anna validar, depois apagar

df_produtos_lancamento['PERIODO'] = pd.to_datetime(df_produtos_lancamento['PERIODO']).dt.strftime('%Y-%m-%d')

df_pivot = (
    df_produtos_lancamento.pivot_table(
        index=['COD_PROD', 'CD: MT', 'CD: NE', 'CD: CO', 'CD: VQ', 'CD: TM'],
        columns='PERIODO',
        values='VALOR',
        aggfunc='sum',
        fill_value=0
    )
    .reset_index()
)

df_pivot.columns.name = None

# gerar excel com df_pivot
df_pivot.to_excel(
    r"C:\Users\carlo\OneDrive\Desktop\PRODUTOS_LANCAMENTO_ANNA_VALIDAR.xlsx",
    index=False,
    float_format="%.2f"
)

In [ ]:
parar_execucao()


# ATENCAO:
# -------------------------------------------------------------------------------------------------
# Próximo Ciclo, desenvolver o ponto de ajuste do programa, que está registrado no arquivo Anotações
# -------------------------------------------------------------------------------------------------


In [19]:
# Criando base de rateio com prepaçao de dados dos ultimos 12 meses de vendas da Krona, para calcular o peso de cada produto na venda total por familia, regional e gestor regional, e criar indices de participação por produto, para usar como base de rateio na distribuição da demanda

#ler parquet df_vendas_krona
df_vendas_krona = pd.read_parquet(pasta_staging_parquet / 'df_vendas_krona.parquet')

# Filtrar df_vendas_krona ultimos 12 meses
df_vendas_krona_12m = df_vendas_krona[df_vendas_krona['PERIODO'] >= (datetime.now() - pd.DateOffset(months=12))]

# Agregar df_vendas_krona_12m por EMPRESA, COD_PROD, DESC_PRODUTO, FAMILIA, LINHA, REGIONAL_GESTOR, REGIONAL, PERIODO, SOMANDO QTD_VENDA E VOL_VENDA
df_vendas_krona_12m_agrupado = df_vendas_krona_12m.groupby(['COD_PROD', 'DESC_PRODUTO', 'FAMILIA', 'LINHA', 'REGIONAL_GESTOR', 'REGIONAL']).agg({'QTD_VENDA': 'sum', 'VOL_VENDA': 'sum'}).reset_index()

# Eliminar linhas com QTD_VENDA
df_vendas_krona_12m_agrupado = df_vendas_krona_12m_agrupado[df_vendas_krona_12m_agrupado['QTD_VENDA'] > 0]

# # Criar coluna divindo VOL_VENDA por QTD_VENDA, nome como PESO_UNIT
df_vendas_krona_12m_agrupado['PESO_UNIT'] = df_vendas_krona_12m_agrupado['VOL_VENDA'] / df_vendas_krona_12m_agrupado['QTD_VENDA']

# Criar DIM_PRODUTO com COD_PROD, DESC_PROD, PESO_UNIT
df_dim_produto = (
    df_vendas_krona_12m_agrupado
    .drop_duplicates(subset=['COD_PROD'])
    [['COD_PROD', 'DESC_PRODUTO', 'PESO_UNIT']]
    .reset_index(drop=True)
)

# # Remover colunas QTD_VENDA e PESO_UNIT do df_vendas_krona_12m_agrupado
df_vendas_krona_12m_agrupado = df_vendas_krona_12m_agrupado.drop(columns=['QTD_VENDA', 'PESO_UNIT'])

# Criar uma coluna TOTAL_VOL_VENDA, somando por REGIONAL_GESTOR, REGIONAL, FAMILIA
df_vendas_krona_12m_agrupado['TOTAL_VOL_VENDA'] = df_vendas_krona_12m_agrupado.groupby(['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA'])['VOL_VENDA'].transform('sum')

# Criar coluna PERC_PART, dividindo VOL_VENDA por TOTAL_VOL_VENDA
df_vendas_krona_12m_agrupado['PERC_PART'] = df_vendas_krona_12m_agrupado['VOL_VENDA'] / df_vendas_krona_12m_agrupado['TOTAL_VOL_VENDA']

In [20]:
# 📥 IMPORTANDO SOMENTE O PLANO REGIONAL (SE HOUVER PLANO CLIENTE, CRIAR UM CÓDIGO)

if copia.exists():
    copia.unlink()

shutil.copy2(origem, copia)

df_plano_regional = pd.read_excel(
    copia,
    sheet_name='PLANO_FINAL_COLAB_REGIONAL',
    engine='pyxlsb'
)

df_plano_regional['PERIODO'] = pd.to_datetime(
    df_plano_regional['PERIODO'],
    unit='D',
    origin='1899-12-30'
)

# Vericar quantas versões existe na coluna REVISAO, e trazer a última
ultima_revisao = df_plano_regional['REVISAO'].max()
df_plano_regional = df_plano_regional[df_plano_regional['REVISAO'] == ultima_revisao]

if copia.exists():
    copia.unlink()

# Agregar REGIONAL_GESTOR, REGIONAL, FAMILIA, PERIODO, SOMANDO DEMANDA
df_plano_regional = df_plano_regional.groupby(['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'PERIODO']).agg({'VALOR': 'sum'}).reset_index()

In [21]:
# Gerando plano detalhando explodindo o plano regional, usando como base de rateio a participação de cada produto na venda total da família, regional e gestor regional, calculada com os dados dos ultimos 12 meses de vendas da Krona, para criar um plano detalhado por produto, com a quantidade em kg e em unidades, para cada período, regional e gestor regional

df_plano_demanda_explodido = df_plano_regional.merge(
    df_vendas_krona_12m_agrupado[
        ['COD_PROD', 'DESC_PRODUTO', 'FAMILIA', 'LINHA', 
         'REGIONAL_GESTOR', 'REGIONAL', 'PERC_PART']
    ],
    on=['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA'],
    how='left'
)

df_plano_demanda_explodido['KG_SKU'] = (
    df_plano_demanda_explodido['VALOR'] *
    df_plano_demanda_explodido['PERC_PART']
)

df_plano_demanda_explodido = df_plano_demanda_explodido.merge(
    df_dim_produto[['COD_PROD', 'PESO_UNIT']],
    on=['COD_PROD'],
    how='left'
)


df_plano_demanda_explodido['QTD_SKU'] = (
    df_plano_demanda_explodido['KG_SKU'] /
    df_plano_demanda_explodido['PESO_UNIT']
)

# Eliminar PERIODOS que a soma das linhas deles for 0
periodos_com_soma_zero = df_plano_demanda_explodido.groupby('PERIODO')['KG_SKU'].sum()
periodos_com_soma_zero = periodos_com_soma_zero[periodos_com_soma_zero == 0].index
df_plano_demanda_explodido = df_plano_demanda_explodido[~df_plano_demanda_explodido['PERIODO'].isin(periodos_com_soma_zero)]

# Eliminar colunas PERC_PART, PESO_UNIT
df_plano_demanda_explodido = df_plano_demanda_explodido.drop(columns=['PERC_PART', 'PESO_UNIT', 'VALOR'])

# Organizar colunas EMPRESA	COD_PROD, DESC_PRODUTO, FAMILIA, LINHA, REGIONAL, REGIONAL_GESTOR, PERIODO, KG_SKU, QTD_SKU
df_plano_demanda_explodido = df_plano_demanda_explodido[
    ['COD_PROD', 'DESC_PRODUTO', 'FAMILIA', 'LINHA', 'REGIONAL_GESTOR', 'REGIONAL', 'PERIODO', 'KG_SKU', 'QTD_SKU']
]

# Salvar na minha DESKTOP um arquivo Excel com o nome PLANO_DEMANDA.xlsx, sem index, e com a formatação de número com vírgula e ponto decimal
df_plano_demanda_explodido.to_excel(
    r"C:\Users\carlo\OneDrive\Desktop\PLANO_DEMANDA.xlsx",
    index=False,
    float_format="%.2f"
)

In [22]:
# Ler Estatístico e colocar no mesmo padrão do plano de demanda para enviar a Anna

# ler parquet df_forecast_vendas_krona_CLIENTE
df_forecast_vendas_krona_CLIENTE = pd.read_parquet(pasta_staging_parquet / 'df_forecast_vendas_krona_CLIENTE.parquet')

# ler parquet df_forecast_vendas_krona_PRODUTO, somente as colunas EMPRESA, COD_PROD, DESC_PRODUTO, FAMILIA, LINHA, REGIONAL, REGIONAL_GESTOR, PERIODO
df_forecast_vendas_krona_PRODUTO = pd.read_parquet(pasta_staging_parquet / 'df_forecast_vendas_krona_PRODUTO.parquet', columns=['EMPRESA', 'COD_PROD', 'DESC_PRODUTO', 'FAMILIA', 'LINHA', 'REGIONAL', 'REGIONAL_GESTOR', 'PERIODO', 'VOL_PREV'])

# Agregar df_forecast_vendas_krona_PRODUTO, COD_PROD, DESC_PRODUTO, FAMILIA, LINHA, REGIONAL_GESTOR, REGIONAL, PERIODO, somando VOL_PREV
df_forecast_vendas_krona_PRODUTO = df_forecast_vendas_krona_PRODUTO.groupby(['COD_PROD', 'DESC_PRODUTO', 'FAMILIA', 'LINHA', 'REGIONAL_GESTOR', 'REGIONAL', 'PERIODO']).agg({'VOL_PREV': 'sum'}).reset_index()

# ler parquet df_dim_peso_unit_vendas
df_dim_peso_unit_vendas = pd.read_parquet(pasta_staging_parquet / 'df_dim_peso_unit_vendas.parquet')

# Remover coluna EMPRESA, e Duplicadas pelo COD_PROD
df_dim_peso_unit_vendas = df_dim_peso_unit_vendas.drop(columns=['EMPRESA']).drop_duplicates(subset=['COD_PROD'])


# Importar PESO_UNIT da DIM PRODUTO
df_forecast_vendas_krona_PRODUTO = df_forecast_vendas_krona_PRODUTO.merge(
    df_dim_peso_unit_vendas[['COD_PROD', 'PESO_UNIT']],
    on=['COD_PROD'],
    how='left'
)

# Eliminar linhas com PESO_UNIT nulo
df_forecast_vendas_krona_PRODUTO = df_forecast_vendas_krona_PRODUTO[df_forecast_vendas_krona_PRODUTO['PESO_UNIT'].notnull()]

# Criar coluna QTD_SKU, dividindo VOL_PREV por PESO_UNIT
df_forecast_vendas_krona_PRODUTO['QTD_SKU'] = df_forecast_vendas_krona_PRODUTO['VOL_PREV'] / df_forecast_vendas_krona_PRODUTO['PESO_UNIT']

# Eliminar colunas PESO_UNIT
df_forecast_vendas_krona_PRODUTO = df_forecast_vendas_krona_PRODUTO.drop(columns=['PESO_UNIT'])

# Renomear coluna VOL_PREV para KG_SKU
df_forecast_vendas_krona_PRODUTO = df_forecast_vendas_krona_PRODUTO.rename(columns={'VOL_PREV': 'KG_SKU'})

# Salvar na minha DESKTOP um arquivo Excel com o nome PLANO_ESTATISTICO.xlsx, sem index, e com a formatação de número com vírgula e ponto decimal
df_forecast_vendas_krona_PRODUTO.to_excel(
    r"C:\Users\carlo\OneDrive\Desktop\PLANO_ESTATISTICO.xlsx",
    index=False,
    float_format="%.2f"
)